# PyRAG on Colab: real models + real retrieval, 5-10 HotpotQA questions

This notebook builds up the pipeline in small, runnable stages:

1. Clone the repo + install lightweight dependencies (no vLLM, no flash-attn)
2. Load a small HotpotQA sample and look at its shape
3. Build a **local E5-base retriever** over each question's own shipped context paragraphs (replaces the paper's Wikipedia-wide server)
4. Load **one shared Qwen2.5-7B-Instruct** in 4-bit via `transformers` (replaces vLLM), used for BOTH the Plan role and the Decompose/Answer roles -- wrapped so it looks exactly like `pyrag.llm.OpenAILLM` to the rest of the code. (We tried two separate models -- Qwen2.5-7B-Instruct + Qwen2.5-Coder-7B-Instruct -- and hit a real GPU out-of-memory error on the free T4; one shared model fits comfortably. The paper's own README allows this "simpler" single-model mode.)
5. Run the real `RAGProgramRunner` end-to-end on 5-10 questions and inspect results

**Runtime:** Colab menu -> Runtime -> Change runtime type -> **T4 GPU** (free tier).

## 1. Setup

We clone the PyRAG repo and install only what we actually need for inference: no `vllm`, no `flash-attn`, no `ray` (those are for the training scripts and won't install cleanly / aren't needed on a free Colab GPU anyway).

In [ ]:
import shutil, os
# Force a clean clone every time -- if a PyRAG folder already exists from a
# previous session (Colab runtimes can persist files across a "Restart session"
# in some cases), `git clone` would otherwise fail silently in a shell-magic
# cell and we'd keep running the OLD code without any obvious error.
shutil.rmtree("PyRAG", ignore_errors=True)
!git clone https://github.com/VIGNESH282006/final-yr-project.git PyRAG
%cd PyRAG

In [ ]:
# Sanity check: print the commit we're actually running. Compare this against
# https://github.com/VIGNESH282006/final-yr-project/commits/main to confirm
# Colab picked up the latest code before trusting any results below.
!git log -1 --oneline

In [ ]:
# transformers/accelerate/bitsandbytes: run the LLMs in-process, 4-bit quantized
# sentence-transformers: run E5-base for retrieval
# datasets: load HotpotQA from Hugging Face
!pip install -q transformers accelerate bitsandbytes sentence-transformers datasets openai

## 2. Load a small HotpotQA sample

HotpotQA (the `distractor` config) ships each question together with 10 context
"documents" -- a mix of the paragraphs that actually contain the answer ("gold"
paragraphs) and unrelated distractor paragraphs, meant to make retrieval non-trivial
even at this small scale. This is exactly what we want for a cheap, self-contained
retrieval corpus: no need to index all of Wikipedia, because the answer-relevant text
is already bundled with the question.

In [ ]:
from datasets import load_dataset

N_SAMPLES = 8  # keep this small (5-10) while we're validating the pipeline
# Note: the short name "hotpot_qa" is no longer resolvable on the HF Hub
# (dataset repo ids now require a namespace/name form) -- this is the current id.
raw = load_dataset("hotpotqa/hotpot_qa", "distractor", split=f"validation[:{N_SAMPLES}]")

for ex in raw:
    print(f"- {ex['question']}  =>  {ex['answer']}")

## 3. Local E5-base retrieval over each question's own context

`pyrag/retrieval_agent.py` defines `RetrievalAgent` as an abstract base class with one
method to implement: `retrieve(query, topk) -> List[str]`. The repo ships two concrete
versions: `HttpRetrievalAgent` (talks to a full Wikipedia-scale server we don't have) and
`MockRetrievalAgent` (a 6-sentence toy corpus, keyword overlap only -- fine for wiring
tests, not real evaluation).

We write a third one, `E5LocalRetrievalAgent`, that:
1. Takes the 10 context paragraphs shipped with ONE HotpotQA question,
2. Embeds each paragraph (and the query) into a vector using **E5-base** -- the same
   retriever family the paper itself uses, just running locally instead of behind a server,
3. Ranks paragraphs by **cosine similarity** (a standard way to measure how "close in
   direction" two vectors are; 1.0 = identical direction, 0 = unrelated, -1 = opposite),
4. Returns the top-k as strings formatted exactly like `HttpRetrievalAgent` does
   (`"Doc N (Title: ...)\n<body>"`), so `pyrag/tools.py` doesn't need to know the
   difference.

E5 models expect a `"query: "` or `"passage: "` prefix on their input text -- that's a
quirk of how E5 was trained, not a general embedding-model rule. We follow it here
because the paper does.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from pyrag.retrieval_agent import RetrievalAgent

_e5_model = SentenceTransformer("intfloat/e5-base-v2")  # downloads once, ~440MB


class E5LocalRetrievalAgent(RetrievalAgent):
    """Retrieves over ONE HotpotQA example's own shipped context paragraphs."""

    def __init__(self, hotpot_example: dict):
        titles = hotpot_example["context"]["title"]
        sentences = hotpot_example["context"]["sentences"]
        self.docs = [
            f"Doc {i+1} (Title: {title})\n{''.join(sents).strip()}"
            for i, (title, sents) in enumerate(zip(titles, sentences))
        ]
        passage_texts = [f"passage: {''.join(sents).strip()}" for sents in sentences]
        # normalize_embeddings=True makes cosine similarity equal to a plain dot product
        self.doc_vectors = _e5_model.encode(passage_texts, normalize_embeddings=True)

    def retrieve(self, query: str, topk: int = 5):
        query_vector = _e5_model.encode(f"query: {query}", normalize_embeddings=True)
        scores = self.doc_vectors @ query_vector  # dot product of normalized vectors = cosine similarity
        top_indices = np.argsort(-scores)[:topk]
        return [self.docs[i] for i in top_indices]

In [ ]:
# Quick sanity check on the first example, before wiring it into the full pipeline.
example0 = raw[0]
retriever0 = E5LocalRetrievalAgent(example0)
print("Question:", example0["question"])
for doc in retriever0.retrieve(example0["question"], topk=3):
    print("---")
    print(doc[:200])

## 4. Load ONE Qwen2.5 model in 4-bit, shared for both roles (no vLLM, no server)

The repo's `pyrag.llm.OpenAILLM` class expects an OpenAI-compatible HTTP server to talk to
(that's what vLLM provides). We don't want to run a server on a single shared Colab GPU, so
instead we load the model directly into this notebook's process with `transformers`.

**Why one model instead of two:** the paper's own README explicitly allows sharing one
model across the Plan / Decompose / Answer roles ("simpler", as an alternative to running
two vLLM instances). Loading BOTH Qwen2.5-7B-Instruct and Qwen2.5-Coder-7B-Instruct at
once hit a real GPU out-of-memory error on the free T4 -- two 7B models in 4-bit, plus
activation memory and KV cache, adds up to more than a T4's ~15GB actually has free after
Colab's own overhead. Using ONE shared model uses roughly half the memory and is
guaranteed to fit comfortably. The trade-off is slightly weaker code generation than the
code-specialized Coder model would give, but the framework's LOGIC (decompose -> plan ->
execute -> confidence-aware retry) is what we're testing, and that doesn't depend on which
model is doing the generating.

**4-bit quantization**: normally each model weight is a 16-bit float. Loading in 4-bit
(via `bitsandbytes`) compresses each weight to ~4 bits, cutting memory to roughly a
quarter, at a small cost in output quality -- what makes a 7B model (normally ~14GB in
16-bit) fit comfortably in a free T4 GPU's VRAM.

We build a small wrapper class, `LocalLLM`, with the exact same `.generate(system_prompt,
user_prompt)` method signature as `pyrag.llm.OpenAILLM`. Because every other file
(`runner.py`, `tools.py`, the agents) only ever calls `llm.generate(...)`, they don't need
to change AT ALL -- we're substituting one implementation of that interface for another.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)


class LocalLLM:
    """Drop-in replacement for pyrag.llm.OpenAILLM that runs the model in-process."""

    def __init__(self, model_name: str, max_new_tokens: int = 1024, temperature: float = 0.7):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=_bnb_config,
            device_map="auto",
        )
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def generate(self, system_prompt: str, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        # apply_chat_template formats the messages the way THIS model was trained to expect
        # (each model family has its own chat markup); tokenize=False so we get plain text back first
        prompt_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(prompt_text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                do_sample=self.temperature > 0,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        # output_ids includes the prompt tokens too; slice them off to keep only the new text
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
# This downloads ~4-5GB once (4-bit) and takes a minute or two.
# ONE model, shared for both the Plan role and the Decompose/Answer roles --
# see the markdown above for why (GPU memory).
shared_llm = LocalLLM("Qwen/Qwen2.5-7B-Instruct")
instruct_llm = shared_llm
plan_llm = shared_llm

In [ ]:
# Sanity check the wrapper works before trusting it inside the full pipeline.
print(instruct_llm.generate("You are a helpful assistant.", "Say hello in one short sentence."))

## 5. Run the real pipeline on our 5-10 questions

This is the exact same `RAGProgramRunner` class used by `main.py` in the repo -- we're
just handing it our `LocalLLM` and `E5LocalRetrievalAgent` instead of the server-backed
versions. One retriever is built per question (since each question has its own 10
context paragraphs).

In [ ]:
from pyrag import RAGProgramRunner

results = []
for i, example in enumerate(raw):
    print(f"\n{'#'*70}\n# Question {i+1}/{len(raw)}: {example['question']}\n{'#'*70}")
    retriever = E5LocalRetrievalAgent(example)
    runner = RAGProgramRunner(llm=instruct_llm, plan_llm=plan_llm, retrieval_agent=retriever)
    result = runner.run(example["question"], topk=3)
    result["gold_answer"] = example["answer"]
    results.append(result)

## 6. Compare predicted vs. gold answers

This is NOT a rigorous Exact Match score yet (that needs the paper's normalization
rules -- lowercasing, punctuation/article stripping, etc. -- which we can add later when
we build proper evaluation). For now this is just a human-readable side-by-side check
that the pipeline produces *sensible* answers on real questions.

In [ ]:
for r in results:
    print(f"Q: {r['original_query']}")
    print(f"  predicted: {r['final_answer']}")
    print(f"  gold     : {r['gold_answer']}")
    print()